# Self-Improving Agents — AI Lab Instructor Notebook

*LLM & Agentic AI · Expert*



**Outcome.** You can simulate a bootstrap loop where verified successful plans become memory that improves future agent behavior.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
from dataclasses import dataclass
from collections import Counter, defaultdict
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Self-Improving Agents — Student Lab

Offline lab: collect verified successful plans, compress them into a simple memory, and compare the agent before and after the update.

## 0 — Task families, tools, and templates

In [ ]:
$10

**Why this works.** # Self-Improving Agents — Instructor Lab

Offline lab: collect verified successful plans, compress them into a simple memory, and compare the agent before and after the update.

## 0 — Task families, tools, and templates

## Task 2: 0 — Stochastic planner

## 0 — Stochastic planner

In [ ]:
def propose_template(task: dict, memory: dict, rng: np.random.Generator) -> str:
    signature = task['signature']
    if signature in memory:
        return memory[signature]
    choices = TEMPLATES[signature]
    probs = BASE_PROBS[signature]
    idx = int(rng.choice(len(choices), p=probs))
    return choices[idx]

name = propose_template(TASKS[0], memory={}, rng=np.random.default_rng(0))
print('template', name)
check('known_template', name in TEMPLATES[TASKS[0]['signature']])

In [ ]:
# Checks
check('known_template', name in TEMPLATES[TASKS[0]['signature']])

## Task 3: 1 — Execute a plan and verify outcome

## 1 — Execute a plan and verify outcome

In [ ]:
$11

In [ ]:
# Checks
check('verify_country', verify_answer(TASKS[0], out))

## Task 4: 2 — Collect successful rollouts

## 2 — Collect successful rollouts

In [ ]:
def collect_successes(tasks, memory: dict, n_rollouts: int = 20, seed: int = 0):
    trial_rng = np.random.default_rng(seed)
    rows = []
    for task in tasks:
        for _ in range(n_rollouts):
            template_name = propose_template(task, memory=memory, rng=trial_rng)
            out = run_template(task, template_name)
            if verify_answer(task, out):
                rows.append({
                    'task_id': task['id'],
                    'signature': task['signature'],
                    'template': template_name,
                })
    return rows

successes = collect_successes(TASKS, memory={}, n_rollouts=25, seed=0)
print('n_successes', len(successes), 'sample', successes[:5])
check('some_successes', len(successes) > 0)

In [ ]:
# Checks
check('some_successes', len(successes) > 0)

## Task 5: 3 — Distill successful traces into memory

## 3 — Distill successful traces into memory

In [ ]:
def update_memory(successes):
    by_signature = defaultdict(list)
    for row in successes:
        by_signature[row['signature']].append(row['template'])
    learned = {}
    for signature, vals in by_signature.items():
        learned[signature] = Counter(vals).most_common(1)[0][0]
    return learned

learned_memory = update_memory(successes)
print('learned_memory', learned_memory)
check('learned_has_country', 'country' in learned_memory)
check('learned_has_sum', 'sum_then_mul' in learned_memory)

In [ ]:
# Checks
check('learned_has_country', 'country' in learned_memory)
check('learned_has_sum', 'sum_then_mul' in learned_memory)

## Task 6: 4 — Evaluate before vs after self-improvement

## 4 — Evaluate before vs after self-improvement

In [ ]:
def evaluate_agent(tasks, memory: dict, tries_per_task: int = 1, seed: int = 0):
    trial_rng = np.random.default_rng(seed)
    oks = 0
    for task in tasks:
        solved = False
        for _ in range(tries_per_task):
            template_name = propose_template(task, memory=memory, rng=trial_rng)
            out = run_template(task, template_name)
            if verify_answer(task, out):
                solved = True
                break
        oks += int(solved)
    return {'success_rate': oks / len(tasks)}

before = np.mean([evaluate_agent(TASKS, memory={}, tries_per_task=1, seed=s)['success_rate'] for s in range(200)])
after = np.mean([evaluate_agent(TASKS, memory=learned_memory, tries_per_task=1, seed=s)['success_rate'] for s in range(200)])
print('before', before, 'after', after)
check('improves', after > before + 0.4)

In [ ]:
# Checks
check('improves', after > before + 0.4)

## Task 7: 5 — Improvement vs extra sampling

## 5 — Improvement vs extra sampling

In [ ]:
before3 = np.mean([evaluate_agent(TASKS, memory={}, tries_per_task=3, seed=s)['success_rate'] for s in range(200)])
after1 = np.mean([evaluate_agent(TASKS, memory=learned_memory, tries_per_task=1, seed=s)['success_rate'] for s in range(200)])
print('before3', before3, 'after1', after1)
check('after1_ge_before3', after1 >= before3)

In [ ]:
# Checks
check('after1_ge_before3', after1 >= before3)